# 03 — Feature Engineering
// Feature extraction & visualization

**Objective**: Extract and analyze features for both Pipeline A and Pipeline B.

Covers:
- MFCC + delta + delta-delta
- Spectral features (centroid, bandwidth, rolloff, contrast, flatness)
- Statistical aggregation
- Mel spectrogram for CNN
- Feature correlation, importance, t-SNE visualization

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

import config
from src.data_loader import load_metadata, load_audio, get_file_path, load_fold_data
from src.dsp_pipeline import pipeline_b_process
from src.feature_extraction import (
    extract_handcrafted_features, extract_mel_spectrogram,
    extract_mfcc, extract_all_features
)

plt.style.use('dark_background')
%matplotlib inline

## 1. Load Data (Fold 1 as sample)

In [ ]:
metadata = load_metadata()
X_raw, y = load_fold_data(metadata, fold_ids=[1])
print(f"Loaded {len(X_raw)} samples from fold 1")

# Apply Pipeline B processing
from tqdm import tqdm
X_processed = np.array([pipeline_b_process(x) for x in tqdm(X_raw, desc='Pipeline B')])

## 2. Extract Features

In [ ]:
# Handcrafted features — Raw
features_raw = extract_all_features(X_raw)
print(f"Raw feature shape: {features_raw.shape}")

# Handcrafted features — DSP processed
features_dsp = extract_all_features(X_processed)
print(f"DSP feature shape: {features_dsp.shape}")

## 3. MFCC Visualization per Class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.patch.set_facecolor('#0a0a0a')

for cls_id, cls_name in enumerate(config.CLASS_NAMES):
    idx = np.where(y == cls_id)[0]
    if len(idx) == 0:
        continue
    sample = X_processed[idx[0]]
    mfcc = extract_mfcc(sample)
    
    ax = axes[cls_id // 5, cls_id % 5]
    ax.imshow(mfcc[:config.N_MFCC], aspect='auto', origin='lower', cmap='magma')
    ax.set_title(cls_name, color='#ffffff', fontsize=9)
    ax.set_facecolor('#111111')

plt.suptitle('MFCC per Class (Pipeline B)', color='#00ffff', fontsize=14)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_mfcc_per_class.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 4. Feature Correlation Heatmap

In [ ]:
# Use first 40 features for readability
corr = np.corrcoef(features_dsp[:, :40].T)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#0a0a0a')
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation (first 40 features)', color='#ffffff')
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_feature_correlation.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=config.RANDOM_SEED, n_jobs=-1)
rf.fit(features_dsp, y)

importances = rf.feature_importances_
top_k = 20
top_indices = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')
ax.barh(range(top_k), importances[top_indices][::-1], color='#00ff41')
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'Feature {i}' for i in top_indices[::-1]], fontsize=8)
ax.set_xlabel('Importance', color='#888888')
ax.set_title(f'Top {top_k} Feature Importances', color='#00ffff')
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_feature_importance.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 6. t-SNE Visualization

In [ ]:
# t-SNE on DSP features
tsne = TSNE(n_components=2, random_state=config.RANDOM_SEED, perplexity=30)
X_tsne = tsne.fit_transform(features_dsp)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')

colors = ['#00ff41', '#00ffff', '#ff0080', '#a855f7', '#ffaa00',
          '#ff4444', '#44ff44', '#4444ff', '#ff44ff', '#ffff44']

for cls_id, cls_name in enumerate(config.CLASS_NAMES):
    mask = y == cls_id
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[cls_id],
               label=cls_name, s=10, alpha=0.7)

ax.set_title('t-SNE — Feature Space (Pipeline B)', color='#00ffff')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_tsne_features.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()